In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "Asia/Shanghai")

# path definition
base_path = "/Volumes/dev/ecommerce"
users_path = f"{base_path}/bronze/users"
products_path = f"{base_path}/bronze/products"
events_path = f"{base_path}/bronze/events"

# dbutils.fs.rm(users_path, True)
# dbutils.fs.rm(products_path, True)
# dbutils.fs.rm(events_path, True)

users = spark.read.format("delta").load(users_path)
products = spark.read.format("delta").load(products_path)
events = spark.read.format("delta").load(events_path)  \
.where("event_date = '2026-04-04'")

users.printSchema()
display(users.count())
users.show(10)
users.orderBy(F.split(F.col("user_id"), "_")[1].cast("int").desc()).show(10)
users.orderBy(F.desc("signup_time")).show(10)

products.printSchema()
display(products.count())
products.orderBy(F.split(F.col("product_id"), "_")[1].cast("int").desc()).show(10)
products.orderBy(F.desc("created_at")).show(10)
products.show(10)
display(products.groupBy("category").count())

events.printSchema()
print("===== BASIC INFO =====")
print("Total rows:", events.count())
print("Date range:", events.select(F.min("event_date"), F.max("event_date")).show())

# =========================================
# 1️⃣ event_id 
# =========================================
dup_event_id = (
    events.groupBy("event_id")
    .count()
    .filter("count > 1")
)
print("Duplicate event_id count:", dup_event_id.count())

# =========================================
# 2️⃣ event_time >= signup_time
# =========================================
invalid_signup = (
    events.join(users, "user_id")
    .filter(F.col("event_time") < F.col("signup_time"))
)
print("event_time < signup_time:", invalid_signup.count())

# =========================================
# 3️⃣ event_time >= product.created_at
# =========================================
invalid_product = (
    events.join(products, "product_id", "left")
    .filter(
        (F.col("product_id").isNotNull()) &
        (F.col("event_time") < F.col("created_at"))
    )
)#.show(10,truncate=False)
#.groupBy(F.to_date(F.col("event_time"))).count().filter("count>1").show(10,truncate=False)
print("event_time < product.created_at:", invalid_product.count())

# =========================================
# 4️⃣ partition
# =========================================
cross_day = events.filter(
    F.to_date("event_time") != F.col("event_date")
)
print("Cross-day events:", cross_day.count())

# =========================================
# 5️⃣ same session time order
# =========================================
from pyspark.sql.window import Window

w = Window.partitionBy("session_id").orderBy("event_time")

order_issue = (
    events.withColumn("prev_time", F.lag("event_time").over(w))
    .filter(F.col("event_time") < F.col("prev_time"))
)
print("Session time order issue:", order_issue.count())

# =========================================
# 6️⃣ deplicate rows
# =========================================
dup_rows = (
    events.groupBy(
        "user_id", "session_id", "product_id",
        "event_time", "event_type"
    )
    .count()
    .filter("count > 1")
)
print("Duplicate rows:", dup_rows.count())

# =========================================
# 7️⃣ funnel 
# =========================================
purchase_without_view = (
    events.filter("event_type = 'purchase'")
    .join(
        events.filter("event_type = 'view'"),
        ["user_id", "product_id"],
        "left_anti"
    )
)
print("Purchase without view:", purchase_without_view.count())

# =========================================
# 8️⃣ session must contain login
# =========================================
session_no_login = (
    events.groupBy("session_id")
    .agg(F.collect_set("event_type").alias("types"))
    .filter(~F.array_contains("types", "login"))
)
print("Sessions without login:", session_no_login.count())

# =========================================
# 9️⃣ session contains logout
# =========================================
session_no_logout = (
    events.groupBy("session_id")
    .agg(F.collect_set("event_type").alias("types"))
    .filter(~F.array_contains("types", "logout"))
)
print("Sessions without logout:", session_no_logout.count())

# =========================================
# 🔟 event_date
# =========================================
events.groupBy("event_date").count().orderBy("event_date").show(50, False)

# =========================================
# 1️⃣1️⃣ event_type distribution
# =========================================
events.groupBy("event_type").count().show()

# =========================================
# 1️⃣2️⃣ device distribution
# =========================================
events.groupBy("device").count().show()

# =========================================
#  duplicated_rows
# =========================================
duplicated_rows = (
    events.groupBy(
        "user_id", "product_id",
        "event_time"
    )
    .count()
    .filter("count > 1")
)
print("Duplicated rows:", duplicated_rows.count())

duplicated_rows = (
    events.groupBy(
        "user_id", "product_id",
        "event_time"
    )
    .agg(F.collect_set("event_type").alias("types"),
         F.count("*").alias("count"))
    .filter("count > 1")
).select("types").distinct().show(truncate=False)


